In [30]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from geopy.distance import geodesic
from scipy.signal import medfilt

def calculate_time_differences_manually(df, datetime_column='datetime', output_column='time_diff'):
    """
    Calculates the time difference (in seconds) between consecutive rows.
    """
    df[datetime_column] = pd.to_datetime(df[datetime_column])
    df = df.sort_values(datetime_column).drop_duplicates(subset=[datetime_column]).reset_index(drop=True)
    datetimes = df[datetime_column].tolist()
    time_diffs = [float('nan')]
    for i in range(1, len(datetimes)):
        delta = datetimes[i] - datetimes[i-1]
        time_diffs.append(delta.total_seconds())
    df[output_column] = time_diffs
    return df

def check_raw_gps_data(df):
    """
    Prints basic diagnostics of the raw GPS data (lat/lon differences and time diff statistics).
    """
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'])
    else:
        print("Datetime column is already in datetime64 format; proceeding as-is.")
    
    df['lat_diff'] = df['lat'].diff().abs()
    df['lon_diff'] = df['lon'].diff().abs()
    
    print("\nFirst 20 rows with differences:")
    # Assumes that calculate_time_differences_manually() has been applied
    print(df[['datetime', 'lat_diff', 'lon_diff', 'time_diff']].iloc[:20])
    print("\nTime Difference Statistics (seconds):")
    print(df['time_diff'].describe())
    
    print("\nPotential Latitude Outliers:")
    print(df[df['lat_diff'] > df['lat_diff'].mean() + 3 * df['lat_diff'].std()][['lat', 'lat_diff', 'datetime']])
    print("\nPotential Longitude Outliers:")
    print(df[df['lon_diff'] > df['lon_diff'].mean() + 3 * df['lon_diff'].std()][['lon', 'lon_diff', 'datetime']])
    print("\nPotential Time Outliers (time differences < 0.5 sec):")
    print(df[df['time_diff'] < 0.5][['datetime', 'time_diff']])
    
    return df

def clean_trajectory(df):
    """
    Performs cleaning on a raw trajectory:
      - Checks that there are enough data points
      - Computes time differences (if not already computed)
      - Computes the next coordinates, geodesic distances, and speeds
      - Applies median filtering to speed
      - Discards trajectories with unrealistic speed profiles
      - Computes bearing and bearing changes
    """
    # Discard if too few rows
    if len(df) < 10:
        print(f"Data discarded: insufficient rows ({len(df)} rows).")
        return pd.DataFrame()
    
    # Ensure time differences are available
    if 'time_diff' not in df.columns:
        df = calculate_time_differences_manually(df)
    
    # Compute next coordinates for distance and bearing calculation
    df['next_lat'] = df['lat'].shift(-1)
    df['next_lon'] = df['lon'].shift(-1)
    
    # Calculate geodesic distance (meters) between consecutive points
    df['distance'] = df.apply(
        lambda row: geodesic((row['lat'], row['lon']), (row['next_lat'], row['next_lon'])).meters
        if pd.notna(row['next_lat']) else np.nan,
        axis=1
    )
    
    # Compute speed (m/s) and smooth using median filter (kernel size=5)
    df['speed'] = df['distance'] / df['time_diff']
    df['speed'] = medfilt(df['speed'], kernel_size=5)
    
    # Basic quality control: reject trajectories with too high mean speed
    mean_speed = df['speed'].mean()
    if mean_speed > 15:
        print(f"Data discarded: mean speed too high ({mean_speed:.2f} m/s).")
        return pd.DataFrame()
    
    # Reject trajectories where more than 50% of rows have zero speed
    zero_speed_count = (df['speed'] == 0).sum()
    if zero_speed_count > len(df) * 0.5:
        print(f"Data discarded: more than 50% of rows have zero speed ({zero_speed_count} rows).")
        return pd.DataFrame()
    
    # Compute bearing between consecutive points and the absolute change in bearing
    df['bearing'] = np.where(
        pd.notna(df['next_lat']),
        compute_bearing(df['lat'], df['lon'], df['next_lat'], df['next_lon']),
        np.nan
    )
    df['bearing_change'] = df['bearing'].diff().abs().fillna(0)
    
    print(f"Data retained: {len(df)} rows after cleaning.")
    return df

# Helper: Compute bearing (used in enrichment)
def compute_bearing(lat1, lon1, lat2, lon2):
    """
    Compute bearing between two GPS coordinates.
    """
    dlon = np.radians(lon2 - lon1)
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.degrees(np.arctan2(y, x)) % 360

In [35]:
from geopy.geocoders import Nominatim

def get_location_name(lat, lon):
    """
    Enriches a GPS coordinate by retrieving a human-readable location name.
    """
    geolocator = Nominatim(user_agent="geoenrichment")
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True)
        return location.address
    except Exception as e:
        return "Unknown Location"

def minimal_angle_diff(diff):
    """Compute the minimal absolute difference between two angles (in degrees)."""
    diff = abs(diff) % 360
    return diff if diff <= 180 else 360 - diff

def compute_enriched_metrics(df):
    """
    Computes enriched trip metrics:
      - Step distances and overall distance (km)
      - Raw speeds with mode-based speed capping and outlier filtering
      - Acceleration (m/s²)
      - Turning metrics (bearing change, turn rate, etc.)
    """
    # Ensure time differences are available
    if 'time_diff' not in df.columns:
        df = calculate_time_differences_manually(df)
    
    # Compute next coordinates and step distance
    df['next_lat'] = df['lat'].shift(-1)
    df['next_lon'] = df['lon'].shift(-1)
    df = df.dropna(subset=['lat', 'lon', 'next_lat', 'next_lon'])
    df['step_distance'] = df.apply(
        lambda row: geodesic((row['lat'], row['lon']), (row['next_lat'], row['next_lon'])).meters,
        axis=1
    )
    total_distance = df['step_distance'].sum() / 1000  # in km
    
    # Compute raw speed (m/s)
    df['speed'] = df['step_distance'] / df['time_diff']
    
    # Apply mode-based speed cap
    if 'transport_mode' in df.columns and not df['transport_mode'].isna().all():
        mode = df['transport_mode'].mode()[0] if not df['transport_mode'].mode().empty else 'unknown'
    else:
        mode = 'unknown'
    
    speed_cap_mps = {
        'walk': 2.78,  # 10 km/h
        'bike': 6.94,  # 25 km/h
        'bus': 11.11,  # 40 km/h
        'car': 13.89,  # 50 km/h
        'taxi': 13.89, # 50 km/h
        'unknown': 13.89
    }.get(mode.lower(), 6.94)  # Default cap
    
    accel_cap_mps2 = {
        'walk': 2.0,
        'bike': 3.0,
        'bus': 3.5,
        'car': 4.0,
        'taxi': 4.0,
        'unknown': 4.0
    }.get(mode.lower(), 4.0)
    
    # Outlier filtering using the IQR method:
    original_speed = df['speed'].copy()
    # Outlier filtering using the IQR method
    Q1 = df['speed'].quantile(0.25)
    Q3 = df['speed'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR

    # Use the lower cap between the IQR upper bound and the mode-based cap
    final_speed_cap = min(upper_bound, speed_cap_mps)
    
    # Count the number of speed outliers before clipping
    outlier_count = (original_speed > final_speed_cap).sum()
    print("Outlier count before clipping:", outlier_count)
    
    df['speed'] = df['speed'].clip(upper=final_speed_cap)
    df['speed'] = df['speed'].clip(upper=speed_cap_mps)
    
    # Smooth speed with a median filter
    df['speed'] = medfilt(df['speed'], kernel_size=5)
    df['speed_kmh'] = df['speed'] * 3.6  # Convert m/s to km/h
    
    # Compute acceleration (m/s²)
    df['acceleration'] = df['speed'].diff() / df['time_diff']
    df['acceleration'] = df['acceleration'].fillna(0)
    
    # # Cap acceleration (if already computed) and speed
    # if 'acceleration' in df.columns:
    #     df['acceleration'] = df['acceleration'].clip(upper=accel_cap_mps2)

    # Turning metrics: Compute bearing and its change using circular difference
    df['bearing'] = df.apply(
        lambda row: compute_bearing(row['lat'], row['lon'], row['next_lat'], row['next_lon']),
        axis=1
    )

    # Use the minimal angle difference to calculate bearing changes
    df['bearing_change'] = df['bearing'].diff().apply(minimal_angle_diff)

    turn_threshold = 30  # degrees
    num_turns = (df['bearing_change'] >= turn_threshold).sum()
    duration_minutes = df['time_diff'].sum() / 60
    turn_rate = num_turns / duration_minutes if duration_minutes > 0 else 0
    avg_turn_angle = df['bearing_change'].mean()
    turn_angle_std = df['bearing_change'].std()
    
    metrics = {
        "total_distance": total_distance,
        "max_speed": df['speed_kmh'].max(),
        "min_speed": df['speed_kmh'].min(),
        "speed_std": df['speed_kmh'].std(),
        "avg_speed": df['speed_kmh'].mean(),
        "avg_acceleration": df['acceleration'].mean(),
        "max_acceleration": df['acceleration'].max(),
        "acceleration_std": df['acceleration'].std(),
        "num_turns": int(num_turns),
        "turn_rate": turn_rate,
        "avg_turn_angle": avg_turn_angle,
        "turn_angle_std": turn_angle_std,
        "avg_bearing_change": avg_turn_angle
    }
    return metrics

def generate_trip_description(df, metrics):
    """
    Generates a human-readable summary of the trip using enriched metrics.
    Also enriches the trip by retrieving start and end location names.
    """
    if df.empty:
        return "No valid data for this trip.", "Unknown"
    
    start = df.iloc[0]
    end = df.iloc[-1]
    
    start_location = get_location_name(start['lat'], start['lon'])
    end_location = get_location_name(end['lat'], end['lon'])
    
    if 'transport_mode' in df.columns and not df['transport_mode'].isna().all():
        mode_series = df['transport_mode']
        mode_value = mode_series.mode()[0]
        mode_count = (mode_series == mode_value).sum()
        total_points = len(mode_series)
        threshold = 0.6
        if (mode_count / total_points) >= threshold:
            transport_mode = mode_value
        else:
            transport_mode = "Mixed"
    else:
        transport_mode = "Unknown"
    
    description = f"""
Trip Summary:
- Start: {start['datetime'].strftime('%Y-%m-%d %H:%M:%S')} at {start_location}
- End: {end['datetime'].strftime('%Y-%m-%d %H:%M:%S')} at {end_location}
- Duration: {(end['datetime'] - start['datetime'])}
- Distance: {metrics['total_distance']:.2f} km
- Average Speed: {metrics['avg_speed']:.2f} km/h
- Average Bearing Change: {metrics['avg_bearing_change']:.2f}°
- Max Speed: {metrics['max_speed']:.2f} km/h
- Min Speed: {metrics['min_speed']:.2f} km/h
- Speed Variability: {metrics['speed_std']:.2f} km/h
- Average Acceleration: {metrics['avg_acceleration']:.2f} m/s²
- Max Acceleration: {metrics['max_acceleration']:.2f} m/s²
- Number of Turns: {metrics['num_turns']}
- Turn Rate: {metrics['turn_rate']:.2f} turns/min
- Average Turn Angle: {metrics['avg_turn_angle']:.2f}°
- Turn Angle Variability: {metrics['turn_angle_std']:.2f}°
- Transport Mode: {transport_mode}
    """
    
    # # Optionally, write the summary to a file.
    # with open("trip_summaries.txt", "a", encoding="utf-8") as f:
    #     f.write(description + "\n\n")
    
    return description, transport_mode

In [36]:
import os
import glob
import geopandas as gpd
import pandas as pd
import warnings
from geopy.distance import geodesic
from scipy.signal import medfilt
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Folder where the cleaned sub-trajectories are stored.
folder_path = "Sub_Trajectories_Cleaned"
geojson_files = glob.glob(os.path.join(folder_path, "**/*.geojson"), recursive=True)
print(f"Number of GeoJSON files found: {len(geojson_files)}")

# Initialize accumulators for global analysis
total_rows_global = 0
extreme_outlier_rows_global = 0
very_extreme_outlier_rows_global = 0
regular_capped_rows_global = 0

# List to store trip-level metrics
trip_data = []
# counter = 0
for file in geojson_files:
    try:
        # counter += 1
        # if counter == 20:
        #     break
        # Read the file into a GeoDataFrame.
        gdf = gpd.read_file(file)
        if gdf.empty:
            print(f"File {file} is empty. Skipping.")
            continue

        # Clean the trajectory using your function.
        gdf_clean = clean_trajectory(gdf)
        if gdf_clean.empty:
            print(f"File {file} did not pass cleaning. Skipping.")
            continue

        # Ensure time differences are computed.
        gdf_clean = calculate_time_differences_manually(gdf_clean)
        if not pd.api.types.is_datetime64_any_dtype(gdf_clean['datetime']):
            gdf_clean['datetime'] = pd.to_datetime(gdf_clean['datetime'])
        gdf_clean = gdf_clean.sort_values('datetime').reset_index(drop=True)

        # Compute necessary step distances for speed calculation
        gdf_clean['next_lat'] = gdf_clean['lat'].shift(-1)
        gdf_clean['next_lon'] = gdf_clean['lon'].shift(-1)
        gdf_clean = gdf_clean.dropna(subset=['lat', 'lon', 'next_lat', 'next_lon'])
        gdf_clean['step_distance'] = gdf_clean.apply(
            lambda row: geodesic((row['lat'], row['lon']), (row['next_lat'], row['next_lon'])).meters,
            axis=1
        )
        
        # Compute raw speeds
        gdf_clean['speed'] = gdf_clean['step_distance'] / gdf_clean['time_diff']

        # Determine mode for capping
        if 'transport_mode' in gdf_clean.columns and not gdf_clean['transport_mode'].isna().all():
            mode = gdf_clean['transport_mode'].mode()[0] if not gdf_clean['transport_mode'].mode().empty else 'unknown'
        else:
            mode = 'unknown'
        
        # Compute the maximum speed for the trip
        trip_max_speed = gdf_clean['speed'].max()
            
        speed_cap_mps = {
            'walk': 2.78,   # 10 km/h
            'bike': 6.94,   # 25 km/h
            'bus': 11.11,   # 40 km/h
            'car': 13.89,   # 50 km/h
            'taxi': 13.89,  # 50 km/h
            'unknown': 13.89
        }.get(mode.lower(), 6.94)

        # Check if the trip's maximum speed exceeds the mode-based cap
        if trip_max_speed >= speed_cap_mps:
            print(f"Trip max speed ({trip_max_speed:.2f} m/s) exceeds the cap of {speed_cap_mps:.2f} m/s.")
        else:
            print(f"Trip max speed ({trip_max_speed:.2f} m/s) is within the cap of {speed_cap_mps:.2f} m/s.")

        # Calculate rows counts before filtering
        initial_rows = len(gdf_clean)
        
        # Count extreme outliers (speed > 2x cap)
        extreme_outliers = (gdf_clean['speed'] > 2 * speed_cap_mps).sum()
        extreme_outlier_rows_global += extreme_outliers
        
        # Count very extreme outliers (speed > 5x cap)
        very_extreme_outliers = (gdf_clean['speed'] > 5 * speed_cap_mps).sum()
        very_extreme_outlier_rows_global += very_extreme_outliers
        
        # Count regular capped points (cap < speed <= 2*cap)
        regular_capped = ((gdf_clean['speed'] > speed_cap_mps) & (gdf_clean['speed'] <= 2 * speed_cap_mps)).sum()
        regular_capped_rows_global += regular_capped
        
        print(f"Initial rows: {initial_rows}")
        print(f"Speed cap: {speed_cap_mps}")
        print(f"Regular capped (cap < speed <= 2x cap): {regular_capped}")
        print(f"Extreme outliers (>2x cap): {extreme_outliers}")
        print(f"Very extreme outliers (>5x cap): {very_extreme_outliers}")
        
        # REMOVE extreme outliers (speed > 2x cap)
        gdf_clean = gdf_clean[gdf_clean['speed'] <= 2 * speed_cap_mps].copy()
        
        # Apply capping for the remaining rows that exceed cap but aren't extreme
        gdf_clean['speed'] = gdf_clean['speed'].clip(upper=speed_cap_mps)
        
        # Update global counter for total rows (after removal)
        total_rows_global += len(gdf_clean)
        
        print(f"Rows after removal: {len(gdf_clean)}")
        print(f"Points removed: {initial_rows - len(gdf_clean)}")
        print(f"Total rows processed globally: {total_rows_global}")

        # Compute enriched metrics (this will perform the capping inside the function)
        metrics = compute_enriched_metrics(gdf_clean)
        if metrics is None:
            print(f"Metrics could not be computed for file {file}. Skipping.")
            continue

        # Calculate additional trip-level information.
        if len(gdf_clean) > 0:  # Check if we still have points after removal
            start_time = gdf_clean['datetime'].iloc[0]
            end_time = gdf_clean['datetime'].iloc[-1]
            duration_sec = (end_time - start_time).total_seconds()

            if 'transport_mode' in gdf_clean.columns:
                mode_series = gdf_clean['transport_mode'].dropna()
                if len(mode_series.unique()) == 1:
                    trip_mode = mode_series.iloc[0]
                else:
                    trip_mode = mode_series.mode()[0]
            else:
                trip_mode = "Unknown"

            summary, _ = generate_trip_description(gdf_clean, metrics)

            # Append additional information to metrics.
            metrics['trip_id'] = os.path.basename(file)
            metrics['start_time'] = start_time
            metrics['end_time'] = end_time
            metrics['duration_sec'] = duration_sec
            metrics['transport_mode'] = trip_mode
            
            # Add outlier info to the metrics
            metrics['initial_points'] = initial_rows
            metrics['points_removed'] = initial_rows - len(gdf_clean)
            metrics['pct_points_removed'] = ((initial_rows - len(gdf_clean)) / initial_rows) * 100 if initial_rows > 0 else 0
            metrics['regular_capped_points'] = regular_capped
            metrics['extreme_outliers_removed'] = extreme_outliers
            metrics['speed_cap'] = speed_cap_mps
            
            metrics['trip_summary'] = summary

            trip_data.append(metrics)

    except Exception as e:
        print(f"Error processing file {file}: {e}")

# Create a DataFrame from the aggregated trip metrics.
df_trip_level = pd.DataFrame(trip_data)
print("Trip-level dataset shape:", df_trip_level.shape)
display(df_trip_level.head())

# Save the aggregated trip-level dataset to CSV.
output_csv = "trip_level_data.csv"
df_trip_level.to_csv(output_csv, index=False)
print(f"Trip-level data saved to '{output_csv}'.")

# Compute global percentages
total_initial_rows = total_rows_global + extreme_outlier_rows_global
if total_initial_rows > 0:
    pct_extreme_outliers_global = (extreme_outlier_rows_global / total_initial_rows) * 100
    pct_very_extreme_outliers_global = (very_extreme_outlier_rows_global / total_initial_rows) * 100
    pct_regular_capped_global = (regular_capped_rows_global / total_initial_rows) * 100
else:
    pct_extreme_outliers_global = 0
    pct_very_extreme_outliers_global = 0
    pct_regular_capped_global = 0

# Write the analysis to a text file in the ML_result folder.
output_analysis_path = os.path.join("ML_result", "capping_analysis.txt")
os.makedirs(os.path.dirname(output_analysis_path), exist_ok=True)  # Ensure the folder exists

with open(output_analysis_path, "w") as f:
    f.write("Global Cleaning Analysis\n")
    f.write("=======================\n")
    f.write(f"Total initial rows: {total_initial_rows}\n")
    f.write(f"Extreme outliers removed (>2x cap): {extreme_outlier_rows_global}\n")
    f.write(f"Percentage of extreme outliers removed: {pct_extreme_outliers_global:.2f}%\n")
    f.write(f"Very extreme outliers subset (>5x cap): {very_extreme_outlier_rows_global}\n")
    f.write(f"Percentage of very extreme outliers: {pct_very_extreme_outliers_global:.2f}%\n\n")
    
    f.write("Remaining Data Analysis\n")
    f.write("=======================\n")
    f.write(f"Total rows after removal: {total_rows_global}\n")
    f.write(f"Rows with speed capped (speed > cap but ≤ 2x cap): {regular_capped_rows_global}\n")
    f.write(f"Percentage of rows capped: {pct_regular_capped_global:.2f}%\n")

print(f"Enhanced cleaning analysis saved to '{output_analysis_path}'.")

Skipping field time: unsupported OGR type: 10


Number of GeoJSON files found: 1092
Data retained: 10 rows after cleaning.
Trip max speed (2.86 m/s) is within the cap of 6.94 m/s.
Initial rows: 9
Speed cap: 6.94
Regular capped (cap < speed <= 2x cap): 0
Extreme outliers (>2x cap): 0
Very extreme outliers (>5x cap): 0
Rows after removal: 8
Points removed: 1
Total rows processed globally: 8
Outlier count before clipping: 0


Skipping field time: unsupported OGR type: 10


Data retained: 825 rows after cleaning.
Trip max speed (54.96 m/s) exceeds the cap of 2.78 m/s.
Initial rows: 824
Speed cap: 2.78
Regular capped (cap < speed <= 2x cap): 30
Extreme outliers (>2x cap): 28
Very extreme outliers (>5x cap): 8
Rows after removal: 795
Points removed: 29
Total rows processed globally: 803
Outlier count before clipping: 52


Skipping field time: unsupported OGR type: 10


...

Data discarded: mean speed too high (16.98 m/s).
File Sub_Trajectories_Cleaned/20080430235534/train_cleaned.geojson did not pass cleaning. Skipping.
Data retained: 223 rows after cleaning.
Trip max speed (15.60 m/s) exceeds the cap of 2.78 m/s.
Initial rows: 222
Speed cap: 2.78
Regular capped (cap < speed <= 2x cap): 0
Extreme outliers (>2x cap): 3
Very extreme outliers (>5x cap): 1
Rows after removal: 218
Points removed: 4
Total rows processed globally: 337673
Outlier count before clipping: 2


Skipping field time: unsupported OGR type: 10


Data retained: 24 rows after cleaning.
Trip max speed (29.80 m/s) exceeds the cap of 2.78 m/s.
Initial rows: 23
Speed cap: 2.78
Regular capped (cap < speed <= 2x cap): 1
Extreme outliers (>2x cap): 5
Very extreme outliers (>5x cap): 2
Rows after removal: 17
Points removed: 6
Total rows processed globally: 337690
Outlier count before clipping: 4
Trip-level dataset shape: (1021, 25)


,total_distance,max_speed,min_speed,speed_std,avg_speed,avg_acceleration,max_acceleration,acceleration_std,num_turns,turn_rate,...,end_time,duration_sec,transport_mode,initial_points,points_removed,pct_points_removed,regular_capped_points,extreme_outliers_removed,speed_cap,trip_summary
0,0.056625,6.202274,1.608878,1.734081,3.870224,0.013033,0.074890,0.047354,0,0.000000,...,2008-11-03 11:02:51,45.0,subway,9,1,11.111111,0,0,6.94,\nTrip Summary:\n- Start: 2008-11-03 11:02:06 ...
1,6.241616,10.008000,0.123001,2.180204,3.425304,0.002405,0.655239,0.068961,454,4.938361,...,2008-11-03 14:35:06,5660.0,walk,824,29,3.519417,30,28,2.78,\nTrip Summary:\n- Start: 2008-11-03 13:00:46 ...
2,3.788475,15.398024,2.016791,3.263822,10.277240,-0.002750,0.022441,0.017267,3,0.122117,...,2008-03-14 09:29:58,1475.0,bike,23,1,4.347826,3,0,6.94,\nTrip Summary:\n- Start: 2008-03-14 09:05:23 ...
3,0.897731,4.839679,0.896842,1.078303,4.320466,0.000126,0.004599,0.002624,6,0.456274,...,2008-03-14 10:48:33,822.0,walk,15,1,6.666667,0,0,2.78,\nTrip Summary:\n- Start: 2008-03-14 10:34:51 ...
4,1.574620,21.707923,1.998040,2.555151,13.062219,-0.005701,0.890108,0.232416,11,1.666667,...,2008-08-02 16:07:55,404.0,bike,186,5,2.688172,5,4,6.94,\nTrip Summary:\n- Start: 2008-08-02 16:01:11 ...


Trip-level data saved to 'trip_level_data.csv'.
Enhanced cleaning analysis saved to 'ML_result/capping_analysis.txt'.


In [ ]:
# import os
# import glob
# import geopandas as gpd
# import pandas as pd
# import warnings
# warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# # Folder where the cleaned sub-trajectories are stored.
# folder_path = "Sub_Trajectories_Cleaned"
# geojson_files = glob.glob(os.path.join(folder_path, "**/*.geojson"), recursive=True)
# print(f"Number of GeoJSON files found: {len(geojson_files)}")

# # List to store trip-level metrics
# trip_data = []

# for file in geojson_files:
#     try:
#         # Read the file into a GeoDataFrame.
#         gdf = gpd.read_file(file)
#         if gdf.empty:
#             print(f"File {file} is empty. Skipping.")
#             continue

#         # Clean the trajectory using your function.
#         gdf_clean = clean_trajectory(gdf)
#         if gdf_clean.empty:
#             print(f"File {file} did not pass cleaning. Skipping.")
#             continue

#         # Ensure time differences are computed.
#         gdf_clean = calculate_time_differences_manually(gdf_clean)
#         # Ensure datetime is properly parsed and sorted.
#         if not pd.api.types.is_datetime64_any_dtype(gdf_clean['datetime']):
#             gdf_clean['datetime'] = pd.to_datetime(gdf_clean['datetime'])
#         gdf_clean = gdf_clean.sort_values('datetime').reset_index(drop=True)

#         # Compute enriched (trip-level) metrics.
#         metrics = compute_enriched_metrics(gdf_clean)
#         if metrics is None:
#             print(f"Metrics could not be computed for file {file}. Skipping.")
#             continue

#         # Calculate additional trip-level information.
#         start_time = gdf_clean['datetime'].iloc[0]
#         end_time = gdf_clean['datetime'].iloc[-1]
#         duration_sec = (end_time - start_time).total_seconds()

#         # Determine overall transport mode:
#         if 'transport_mode' in gdf_clean.columns:
#             mode_series = gdf_clean['transport_mode'].dropna()
#             if len(mode_series.unique()) == 1:
#                 trip_mode = mode_series.iloc[0]
#             else:
#                 trip_mode = mode_series.mode()[0]
#         else:
#             trip_mode = "Unknown"

#         # Optionally, generate a human-readable trip summary.
#         summary, _ = generate_trip_description(gdf_clean, metrics)

#         # Add extra fields to the metrics dictionary.
#         metrics['trip_id'] = os.path.basename(file)
#         metrics['start_time'] = start_time
#         metrics['end_time'] = end_time
#         metrics['duration_sec'] = duration_sec
#         metrics['transport_mode'] = trip_mode
#         metrics['trip_summary'] = summary

#         trip_data.append(metrics)

#     except Exception as e:
#         print(f"Error processing file {file}: {e}")

# # Create a DataFrame from the aggregated trip metrics.
# df_trip_level = pd.DataFrame(trip_data)
# print("Trip-level dataset shape:", df_trip_level.shape)
# display(df_trip_level.head())

# # Save the aggregated trip-level dataset to CSV.
# output_csv = "trip_level_data.csv"
# df_trip_level.to_csv(output_csv, index=False)
# print(f"Trip-level data saved to '{output_csv}'.")

Skipping field time: unsupported OGR type: 10


Number of GeoJSON files found: 1096
Data retained: 15 rows after cleaning.
Trip max speed (0.00 m/s) is within the cap of 6.94 m/s.
Speed cap: 6.94
Speed: 0     NaN
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
Name: speed, dtype: float64
rows_exceed_mode: 0
mode_capped_rows_global: 0


Skipping field time: unsupported OGR type: 10


Data retained: 900 rows after cleaning.
Trip max speed (0.00 m/s) is within the cap of 2.78 m/s.
Speed cap: 2.78
Speed: 0      NaN
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
895    0.0
896    0.0
897    0.0
898    0.0
899    0.0
Name: speed, Length: 900, dtype: float64
rows_exceed_mode: 0
mode_capped_rows_global: 0


Skipping field time: unsupported OGR type: 10


...

Data retained: 568 rows after cleaning.
Trip max speed (0.00 m/s) is within the cap of 6.94 m/s.
Speed cap: 6.94
Speed: 0      NaN
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
563    0.0
564    0.0
565    0.0
566    0.0
567    0.0
Name: speed, Length: 568, dtype: float64
rows_exceed_mode: 0
mode_capped_rows_global: 0


Skipping field time: unsupported OGR type: 10


Data retained: 698 rows after cleaning.
Trip max speed (0.00 m/s) is within the cap of 11.11 m/s.
Speed cap: 11.11
Speed: 0      NaN
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
693    0.0
694    0.0
695    0.0
696    0.0
697    0.0
Name: speed, Length: 698, dtype: float64
rows_exceed_mode: 0
mode_capped_rows_global: 0
Trip-level dataset shape: (19, 19)


,total_distance,max_speed,min_speed,speed_std,avg_speed,avg_acceleration,max_acceleration,acceleration_std,num_turns,turn_rate,avg_turn_angle,turn_angle_std,avg_bearing_change,trip_id,start_time,end_time,duration_sec,transport_mode,trip_summary
0,0.099576,6.804614,0.0,2.175556,4.548674,0.007313,0.128242,0.075961,3,2.769231,44.343175,58.642820,44.343175,subway_cleaned.geojson,2008-11-03 11:01:56,2008-11-03 11:03:06,70.0,subway,\nTrip Summary:\n- Start: 2008-11-03 11:01:56 ...
1,7.335194,10.008000,0.0,2.491312,3.861793,0.002583,0.655239,0.076420,492,5.210944,52.426213,49.501440,52.426213,walk_cleaned.geojson,2008-11-03 13:00:41,2008-11-03 14:35:11,5670.0,walk,\nTrip Summary:\n- Start: 2008-11-03 13:00:41 ...
2,4.270492,24.984000,0.0,4.170243,9.485650,0.004165,0.144995,0.029646,6,0.223602,22.217181,33.085706,22.217181,bike_cleaned.geojson,2008-03-14 09:03:40,2008-03-14 09:31:28,1668.0,bike,\nTrip Summary:\n- Start: 2008-03-14 09:03:40 ...
3,1.028568,5.929675,0.0,1.576715,4.152453,-0.000010,0.007549,0.004052,8,0.561404,46.821988,42.737292,46.821988,walk_cleaned.geojson,2008-03-14 10:34:18,2008-03-14 10:48:47,869.0,walk,\nTrip Summary:\n- Start: 2008-03-14 10:34:18 ...
4,1.682420,22.198744,0.0,3.230470,13.281676,-0.004001,1.237699,0.248023,18,2.583732,14.340835,19.533411,14.340835,bike_cleaned.geojson,2008-08-02 16:00:57,2008-08-02 16:07:57,420.0,bike,\nTrip Summary:\n- Start: 2008-08-02 16:00:57 ...


Trip-level data saved to 'trip_level_data.csv'.
Capping analysis saved to 'ML_result/capping_analysis.txt'.
